# Tutorial 7: Smart-meter analytics and machine-learning load forecasting

This tutorial connects smart-meter exploration with day-ahead load forecasting at two scales:

- the **GEFCom2012 aggregate system load**, where regular patterns are comparatively clear; and
- one **Building Data Genome 2 education building**, where operation is more variable.

The emphasis is on transparent, origin-valid features and simple machine-learning pipelines. The aim is not to build a production forecasting system.

## Learning outcomes

By the end of the tutorial, you should be able to:

1. explore hourly meter data and identify patterns or data-quality concerns;
2. formulate day-ahead forecasting as supervised learning;
3. distinguish predictors available at the forecast origin from future information;
4. compare pooled Ridge regression, hour-specific Ridge models, a decision tree and a random forest with seasonal-naïve benchmarks;
5. reshape an hourly series into daily rows and train one model for each hour of the day;
6. evaluate day-ahead forecasts over a chronological test set; and
7. explain why aggregate and individual-building loads behave differently.

## Indicative schedule

- set-up and smart-meter exploration: 20 minutes;
- forecast formulation and features: 20 minutes;
- benchmarks and machine-learning models: 25 minutes;
- hour-specific linear models and test-set evaluation: 25 minutes;
- building-level transfer and discussion: 10 minutes.

## 1. Set-up and data loading

The tutorial uses compact extracts stored in `data/tutorial_07`. Run the notebook from the repository root. The data-loading code is supplied.

The aggregate series is `Z21` from GEFCom2012. The building series is the electricity meter for `Rat_education_Alfonso`, a K–12 education building at the `Rat` site in Building Data Genome 2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_DIR = Path("data") / "tutorial_07"

system_data = pd.read_csv(
    DATA_DIR / "gefcom2012_system_load.csv",
    index_col="timestamp",
    parse_dates=True,
).sort_index()

building_data = pd.read_csv(
    DATA_DIR / "building_data_genome_sample.csv",
    index_col="timestamp",
    parse_dates=True,
).sort_index()

print("System data:", system_data.shape)
print("Building data:", building_data.shape)

### Data-quality overview

The following summary is supplied so that we can focus on interpreting the data rather than writing another checking function. Remember that a row with `NaN` differs from a timestamp that is absent from the index.

In [ ]:
for name, data in {"GEFCom system": system_data, "BDG2 building": building_data}.items():
    expected_index = pd.date_range(data.index.min(), data.index.max(), freq="h")
    missing_timestamps = expected_index.difference(data.index)
    print(f"\n{name}")
    print(f"  period: {data.index.min()} to {data.index.max()}")
    print(f"  duplicate timestamps: {data.index.duplicated().sum()}")
    print(f"  absent hourly timestamps: {len(missing_timestamps)}")
    print("  NaN values by column:")
    print(data.isna().sum().to_string())

## 2. Task 1 — Explore the two load series

First inspect two weeks of each series. Then compare their typical daily profiles and visualise the building data as a day–hour heatmap.

### 1.1 Two-week traces

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

system_data.loc["2007-10-01":"2007-10-14", "load_mw"].plot(
    ax=axes[0], color="tab:blue", linewidth=1.2
)
axes[0].set(title="GEFCom2012 aggregate load: two weeks", ylabel="Load (MW)", xlabel="")

building_data.loc["2017-10-01":"2017-10-14", "electricity_kwh"].plot(
    ax=axes[1], color="tab:orange", linewidth=1.2
)
axes[1].set(
    title="Building Data Genome education building: two weeks",
    ylabel="Electricity use (kWh per hour)",
    xlabel="Timestamp",
)

plt.tight_layout()
plt.show()

### 1.2 Typical daily profiles

Calculate mean load by hour of day. For the building, separate weekdays and weekends.

In [ ]:
# TODO 1.2: calculate the mean system profile by hour of day.
system_profile = ...

# TODO 1.2: add a Weekday/Weekend label, then calculate the building profile.
building_for_profile = ...
building_profile = ...

if system_profile is ... or building_profile is ...:
    raise NotImplementedError("Complete the hourly profile calculations in Task 1.2.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
system_profile.plot(ax=axes[0], marker="o", color="tab:blue")
axes[0].set(title="Mean system profile", xlabel="Hour of day", ylabel="Load (MW)")

building_profile.plot(ax=axes[1], marker="o")
axes[1].set(
    title="Mean building profile",
    xlabel="Hour of day",
    ylabel="Electricity use (kWh per hour)",
)
plt.tight_layout()
plt.show()

### 1.3 Building day–hour heatmap

Create one row per date and one column per hour. Look for weekday/weekend structure, closures and unusual days.

In [ ]:
heatmap_source = building_data.loc["2017-09-18":"2017-10-15", ["electricity_kwh"]].copy()

# TODO 1.3: add date and hour columns, then pivot to date × hour.
building_heatmap = ...

if building_heatmap is ...:
    raise NotImplementedError("Create the date-by-hour table in Task 1.3.")

fig, ax = plt.subplots(figsize=(11, 6))
image = ax.imshow(building_heatmap, aspect="auto", cmap="viridis")
ax.set(title="Building electricity use by date and hour", xlabel="Hour of day", ylabel="Date")
ax.set_xticks(range(0, 24, 3))
ax.set_yticks(range(0, len(building_heatmap), 4))
ax.set_yticklabels([building_heatmap.index[i] for i in range(0, len(building_heatmap), 4)])
fig.colorbar(image, ax=ax, label="Electricity use (kWh per hour)")
plt.tight_layout()
plt.show()

### 1.4 Weather and variability

Plot electricity use against air temperature and calculate the coefficient of variation, defined as standard deviation divided by mean. This is only an illustration: the two datasets also differ in climate, use and measurement context, so scale alone cannot explain every difference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(
    system_data["air_temperature_c"], system_data["load_mw"],
    s=5, alpha=0.18, color="tab:blue"
)
axes[0].set(xlabel="Air temperature (°C)", ylabel="Load (MW)", title="Aggregate system")

axes[1].scatter(
    building_data["air_temperature_c"], building_data["electricity_kwh"],
    s=5, alpha=0.18, color="tab:orange"
)
axes[1].set(
    xlabel="Air temperature (°C)",
    ylabel="Electricity use (kWh per hour)",
    title="Education building",
)
plt.tight_layout()
plt.show()

# TODO 1.4: calculate standard deviation / mean for each load series.
variability = ...

if variability is ...:
    raise NotImplementedError("Calculate the coefficients of variation in Task 1.4.")
variability.to_frame(name="coefficient_of_variation").round(3)

**Discuss briefly**

1. Which series has the clearer daily cycle?
2. What differences do you see between building weekdays and weekends?
3. Which dates in the heatmap deserve further investigation?
4. Why should a low or zero building reading not automatically be treated as genuine low demand?

## 3. Task 2 — Define an origin-valid day-ahead forecast

For each day in the test period, we issue a forecast at the end of the preceding day for all 24 hours of the next day. We then move the origin forward by 24 hours and repeat.

```text
observed history                    forecast day
... D-2 ... | ... D-1 ... 23:00 || D 00:00 ........ D 23:00
                              origin   lead 1 ........ lead 24
```

At the origin we know:

- past load and weather observations;
- the target timestamps and their calendar information; and
- any genuine weather forecast issued before the origin.

We do **not** know realised target-day load or realised target-day weather.

> Operational note: day-ahead electricity forecasts are commonly issued around 11:00–12:00 on day D−1. Using the end of D−1 here is a simplification that keeps every target day aligned to a 24-hour block.

For this exercise, yesterday's same-hour temperature is used as a transparent weather-persistence proxy. It is not a substitute for a real numerical weather forecast.

**Question:** Why would `load_lag_1` be invalid for most hours of a 24-hour fixed-origin forecast, even though it is a past value in an ordinary one-step-ahead dataset?

## 4. Task 3 — Construct forecasting features

The feature function is intentionally short. Every row represents a target timestamp. The load and temperature lags refer to observations available before the corresponding day-ahead origin.

Complete the lagged features and the calendar variables. We retain integer hour and day-of-week variables for the random forest, and cyclical versions for the pooled Ridge and single-tree models. Do not use centred windows or contemporaneous realised temperature.

In [ ]:
def make_features(data, target_column, temperature_column):
    # Return one row of origin-valid predictors for each target timestamp.
    features = pd.DataFrame(index=data.index)
    features["target"] = data[target_column]

    # TODO 3.1: add load lags of 24 and 168 hours.
    # TODO 3.2: use the previous day's same-hour temperature as a persistence proxy.

    hour = features.index.hour
    day_of_week = features.index.dayofweek

    # TODO 3.3: retain hour and day_of_week as integer columns.
    # TODO 3.4: add sine and cosine encodings for hour and day of week.
    # TODO 3.5: add an integer weekend indicator.

    required = {
        "load_lag_24", "load_lag_168", "temperature_lag_24",
        "hour", "day_of_week", "hour_sin", "hour_cos",
        "day_sin", "day_cos", "is_weekend",
    }
    missing = required.difference(features.columns)
    if missing:
        raise NotImplementedError(f"Complete make_features. Missing: {sorted(missing)}")
    return features.dropna()

In [ ]:
BASE_FEATURES = [
    "load_lag_24",
    "load_lag_168",
    "temperature_lag_24",
    "is_weekend",
]

CYCLICAL_FEATURES = BASE_FEATURES + [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

INTEGER_FEATURES = BASE_FEATURES + ["hour", "day_of_week"]

# The pooled Ridge, single tree and building example use cyclical features.
FEATURE_COLUMNS = CYCLICAL_FEATURES

# The hour-specific Ridge model uses daily profiles of these lagged inputs.
# It does not need explicit hour or calendar encodings.
HOURLY_RIDGE_INPUTS = [
    "load_lag_24",
    "load_lag_168",
    "temperature_lag_24",
]

system_features = make_features(system_data, "load_mw", "air_temperature_c")
system_features.head()

### Chronological split for GEFCom2012

- training: 8 January to 30 June 2007;
- validation: July 2007;
- test: 1–28 August 2007.

The validation month lets us check the modelling choices before opening the test period. The final test forecasts use models refitted on training plus validation data.

In [ ]:
TRAIN_START = pd.Timestamp("2007-01-08 00:00")
TRAIN_END = pd.Timestamp("2007-06-30 23:00")
VALIDATION_START = pd.Timestamp("2007-07-01 00:00")
VALIDATION_END = pd.Timestamp("2007-07-31 23:00")
TEST_START = pd.Timestamp("2007-08-01 00:00")
TEST_END = pd.Timestamp("2007-08-28 23:00")

train = system_features.loc[TRAIN_START:TRAIN_END]
validation = system_features.loc[VALIDATION_START:VALIDATION_END]
test = system_features.loc[TEST_START:TEST_END]

assert len(validation) == 31 * 24
assert len(test) == 28 * 24
assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()

pd.Series({"training rows": len(train), "validation rows": len(validation), "test rows": len(test)})

## 5. Task 4 — Seasonal-naïve benchmarks and metrics

Create daily and weekly seasonal-naïve forecasts for the validation month. Because the corresponding lag columns contain only past observations, no fitted function is needed.

In [ ]:
validation_forecasts = pd.DataFrame(index=validation.index)
validation_forecasts["Actual"] = validation["target"]

# TODO 4.1: add the daily and weekly seasonal-naïve forecasts.

required = {"Daily seasonal naive", "Weekly seasonal naive"}
missing = required.difference(validation_forecasts.columns)
if missing:
    raise NotImplementedError(f"Add the benchmark forecasts. Missing: {sorted(missing)}")
validation_forecasts.head()

The scoring helper is supplied. MASE uses the mean absolute **weekly seasonal difference in the training series** as its scale:

\[
\mathrm{MASE}=\frac{\mathrm{mean}(|y_t-\hat y_t|)}
{\mathrm{mean}(|y_t-y_{t-168}|)\text{ in training})}.
\]

The denominator is a training-data benchmark. A weekly seasonal-naïve forecast evaluated over a different validation or test period is therefore not guaranteed to have MASE exactly equal to one.

In [ ]:
def forecast_scores(actual, forecast, training_series, m=168):
    if isinstance(actual, pd.Series) and isinstance(forecast, pd.Series):
        if not actual.index.equals(forecast.index):
            raise ValueError("Actual and forecast indices must match.")
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    if len(actual) != len(forecast):
        raise ValueError("Actual and forecast must have the same length.")

    errors = actual - forecast
    # Keep missing rows in place so that diff(m) still means exactly m hours.
    training_series = pd.Series(training_series, dtype=float)
    scale = training_series.diff(m).abs().dropna().mean()
    if not np.isfinite(scale) or scale == 0:
        raise ValueError("The MASE scaling denominator is invalid or zero.")

    return {
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "MASE (m=168)": np.mean(np.abs(errors)) / scale,
    }

In [ ]:
# TODO 4.2: call forecast_scores for each benchmark and assemble a table.
validation_scores = ...

if validation_scores is ...:
    raise NotImplementedError("Create the validation benchmark table in Task 4.2.")
validation_scores.round(2)

## 6. Task 5 — Fit simple machine-learning models

We begin with three models:

- **pooled Ridge regression:** one linear model for all hours, using standardised cyclical predictors;
- **a decision tree:** a single nonlinear model using the same cyclical predictors; and
- **a random forest:** an average of many trees, using integer hour and day-of-week variables directly.

The model settings are specified rather than tuned extensively. A preliminary validation comparison found that the integer calendar variables worked slightly better for this random forest than cyclical or one-hot representations. We therefore retain only the simpler integer version here and keep the tutorial focused on forecasting.

In [ ]:
# TODO 5.1: create a StandardScaler + Ridge pipeline with alpha=10.
ridge_model = ...

# TODO 5.2: create a DecisionTreeRegressor using:
# max_depth=8, min_samples_leaf=20, random_state=42.
tree_model = ...

# TODO 5.3: create a RandomForestRegressor using:
# n_estimators=150, max_depth=12, min_samples_leaf=4,
# max_features=0.8, random_state=42 and n_jobs=1.
rf_model = ...

if ridge_model is ... or tree_model is ... or rf_model is ...:
    raise NotImplementedError("Create all three models in Task 5.")

# TODO 5.4: fit the models on train and add their validation forecasts.
# Use FEATURE_COLUMNS for Ridge and the tree, but INTEGER_FEATURES for the forest.

### Task 5.5 — One Ridge model for each hour of the day

The pooled Ridge model assumes that one set of coefficients is suitable for every hour. A simple alternative is to reshape the data so that:

- each **row** represents one day;
- the 24 target **columns** represent hours 00:00 to 23:00; and
- the inputs contain the preceding-day load profile, preceding-week load profile and preceding-day temperature profile.

We then fit 24 small Ridge models: target column 0 has its own model, target column 1 has another, and so on. Explicit hour-of-day and calendar-seasonality terms are omitted because each model always predicts the same hour, while the lagged profiles already carry recent daily and weekly information. This arrangement is also useful for problems such as day-ahead solar forecasting, where each daylight hour can have a different relationship with its predictors.

In [ ]:
def reshape_complete_days(feature_data, predictor_columns):
    '''Return daily predictor and target matrices from complete hourly days.'''
    if len(feature_data) % 24 != 0:
        raise ValueError("The selected interval must contain complete 24-hour days.")
    if feature_data.index[0].hour != 0 or feature_data.index[-1].hour != 23:
        raise ValueError("The interval must start at 00:00 and end at 23:00.")

    expected_index = pd.date_range(
        feature_data.index[0], feature_data.index[-1], freq="h"
    )
    if not feature_data.index.equals(expected_index):
        raise ValueError("The interval must have a continuous hourly index.")

    predictor_blocks = [
        feature_data[column].to_numpy().reshape(-1, 24)
        for column in predictor_columns
    ]
    X_daily = np.hstack(predictor_blocks)
    y_daily = feature_data["target"].to_numpy().reshape(-1, 24)
    return X_daily, y_daily


def fit_hourly_ridge_models(X_daily, y_daily, alpha=10.0):

    models = []
    for hour in range(24):
        # TODO 5.5a: create a scaled Ridge pipeline, fit it to
        # y_daily[:, hour], and append it to models.
        pass

    if len(models) != 24:
        raise NotImplementedError("Fit one Ridge model for each target hour.")
    return models


def predict_hourly_ridge(models, X_daily):
    '''Predict all 24 hourly columns and return a daily matrix.'''
    # TODO 5.5b: predict with every model and join the columns with
    # np.column_stack.
    raise NotImplementedError("Return the 24-column prediction matrix.")


X_train_daily, y_train_daily = reshape_complete_days(
    train, HOURLY_RIDGE_INPUTS
)
X_validation_daily, y_validation_daily = reshape_complete_days(
    validation, HOURLY_RIDGE_INPUTS
)

hourly_ridge_models = fit_hourly_ridge_models(
    X_train_daily, y_train_daily, alpha=10.0
)
hourly_ridge_validation = predict_hourly_ridge(
    hourly_ridge_models, X_validation_daily
)

# Flatten row by row to recover the original chronological hourly order.
validation_forecasts["Hourly Ridge"] = hourly_ridge_validation.reshape(-1)

In [ ]:
# TODO 5.6: rebuild the validation score table with both benchmarks and
# all four machine-learning methods.
validation_scores = ...

if validation_scores is ...:
    raise NotImplementedError("Create the complete validation comparison table.")
validation_scores.round(2)

## 7. Task 6 — Forecast the complete test set

Refit the models on the combined training and validation period, then predict the complete four-week test table directly. Scikit-learn can predict all test rows in one call, and the hour-specific Ridge models predict the complete daily matrix at once.

This vectorised calculation still represents repeated day-ahead forecasts. For each target day, `load_lag_24`, `load_lag_168` and `temperature_lag_24` contain only information that would be available at that day's forecast origin. We are **not** claiming that all four weeks were forecast from one origin.

In [ ]:
development = system_features.loc[TRAIN_START:VALIDATION_END]

# TODO 6.1: refit the pooled Ridge, tree and random forest on development.
# Remember that the forest uses INTEGER_FEATURES.

X_development_daily, y_development_daily = reshape_complete_days(
    development, HOURLY_RIDGE_INPUTS
)
X_test_daily, y_test_daily = reshape_complete_days(test, HOURLY_RIDGE_INPUTS)

# TODO 6.2: refit the 24 hour-specific Ridge models on the development
# matrices and predict X_test_daily.

system_forecasts = pd.DataFrame(index=test.index)
system_forecasts["Actual"] = test["target"]
system_forecasts["Daily seasonal naive"] = test["load_lag_24"]
system_forecasts["Weekly seasonal naive"] = test["load_lag_168"]

# TODO 6.3: add the pooled Ridge, hourly Ridge, tree and forest forecasts.
# Flatten the hourly Ridge matrix row by row with .reshape(-1).

required = {
    "Actual", "Daily seasonal naive", "Weekly seasonal naive",
    "Pooled Ridge", "Hourly Ridge", "Tree", "Random forest",
}
missing = required.difference(system_forecasts.columns)
if missing:
    raise NotImplementedError(f"Complete the test forecasts. Missing: {sorted(missing)}")
assert len(system_forecasts) == 28 * 24
system_forecasts.head()

In [ ]:
# TODO 6.4: compare both benchmarks and all four ML methods over the test set.
system_scores = ...

if system_scores is ...:
    raise NotImplementedError("Create the full test-set comparison table.")
system_scores.round(2)

In [ ]:
plot_week = system_forecasts.loc["2007-08-06":"2007-08-12"]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
plot_week[["Actual", "Daily seasonal naive", "Weekly seasonal naive"]].plot(ax=axes[0])
axes[0].set(title="Aggregate load: seasonal benchmarks", ylabel="Load (MW)", xlabel="")

plot_week[
    ["Actual", "Pooled Ridge", "Hourly Ridge", "Tree", "Random forest"]
].plot(ax=axes[1])
axes[1].set(title="Aggregate load: machine-learning models", ylabel="Load (MW)", xlabel="Timestamp")

plt.tight_layout()
plt.show()

## 8. Task 7 — Transfer the workflow to an individual building

Now reuse the pooled Ridge and tree models for the education building. The building split is:

- training: 8 January 2016 to 31 August 2017;
- validation: September 2017;
- test: 1–28 October 2017.

Rows affected by missing target, lag or weather-proxy values are excluded by `make_features`. The October test period is complete after feature construction.

Unlike a seasonal-naïve forecast or a tree trained on non-negative observations, an unconstrained regression model can produce negative values. Negative electricity use is not physically meaningful, so after checking how often it occurs we apply the simple projection

\[
\hat{y}^{+}_t = \max(0, \hat{y}_t).
\]

The check is important: projection should be an explicit modelling decision, not a hidden cleaning step.

In [ ]:
building_features = make_features(
    building_data,
    target_column="electricity_kwh",
    temperature_column="air_temperature_c",
)

BUILDING_TRAIN_START = pd.Timestamp("2016-01-08 00:00")
BUILDING_TRAIN_END = pd.Timestamp("2017-08-31 23:00")
BUILDING_VALIDATION_START = pd.Timestamp("2017-09-01 00:00")
BUILDING_VALIDATION_END = pd.Timestamp("2017-09-30 23:00")
BUILDING_TEST_START = pd.Timestamp("2017-10-01 00:00")
BUILDING_TEST_END = pd.Timestamp("2017-10-28 23:00")

building_development = building_features.loc[
    BUILDING_TRAIN_START:BUILDING_VALIDATION_END
]
building_test = building_features.loc[BUILDING_TEST_START:BUILDING_TEST_END]
building_mase_training_series = building_data.loc[
    BUILDING_TRAIN_START:BUILDING_VALIDATION_END, "electricity_kwh"
]

assert len(building_test) == 28 * 24
assert not building_test.isna().any().any()

In [ ]:
# TODO 7.1: create new Ridge and tree models using the same settings as before.
building_ridge = ...
building_tree = ...

if building_ridge is ... or building_tree is ...:
    raise NotImplementedError("Create the two building models in Task 7.")

# TODO 7.2: fit both models on building_development.

building_forecasts = pd.DataFrame(index=building_test.index)
building_forecasts["Actual"] = building_test["target"]
building_forecasts["Daily seasonal naive"] = building_test["load_lag_24"]
building_forecasts["Weekly seasonal naive"] = building_test["load_lag_168"]

# TODO 7.3: predict the complete test table with Ridge and the tree.

required = {"Actual", "Daily seasonal naive", "Weekly seasonal naive", "Ridge", "Tree"}
missing = required.difference(building_forecasts.columns)
if missing:
    raise NotImplementedError(f"Complete the building forecasts. Missing: {sorted(missing)}")

# Count negative values before correcting them, then project all forecasts to zero.
building_model_columns = building_forecasts.columns.drop("Actual")
negative_forecast_counts = (
    building_forecasts[building_model_columns] < 0
).sum()
print("Negative forecasts before projection:")
display(negative_forecast_counts.to_frame("Count"))

building_forecasts.loc[:, building_model_columns] = building_forecasts[
    building_model_columns
].clip(lower=0)
assert (building_forecasts[building_model_columns] >= 0).all().all()

# TODO 7.4: create the building metric table. Use
# building_mase_training_series for the MASE denominator.
building_scores = ...

if building_scores is ...:
    raise NotImplementedError("Create the building comparison table in Task 7.4.")
building_scores.round(2)

In [ ]:
building_plot_week = building_forecasts.loc["2017-10-09":"2017-10-15"]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
building_plot_week[["Actual", "Daily seasonal naive", "Weekly seasonal naive"]].plot(ax=axes[0])
axes[0].set(
    title="Education building: seasonal benchmarks",
    ylabel="Electricity use (kWh per hour)",
    xlabel="",
)

building_plot_week[["Actual", "Ridge", "Tree"]].plot(ax=axes[1])
axes[1].set(
    title="Education building: machine-learning models",
    ylabel="Electricity use (kWh per hour)",
    xlabel="Timestamp",
)
plt.tight_layout()
plt.show()

## 9. Interpretation questions

Answer each question in two or three sentences.

1. Which method performs best for aggregate load under MAE, RMSE and MASE?
2. How does the hour-specific Ridge benchmark compare with the pooled Ridge model?
3. Why are explicit hour-of-day variables unnecessary in the hour-specific Ridge models?
4. Why can the complete test table be predicted in one call without implying a four-week-ahead forecast from one origin?
5. Does the same forecasting method perform best for the education building?
6. Why is the weekly seasonal-naïve model a compulsory benchmark here?
7. What exactly is the reference forecast in the MASE denominator? Why need the test-set weekly forecast not have MASE equal to one?
8. Why might aggregation make daily and weekly behaviour easier to forecast?
9. Why do we check for negative building forecasts, and what does projection to zero do?
10. Which feature is only a weather-forecast proxy? How would you improve it operationally?
11. Identify one apparently sensible predictor that would leak future information in this day-ahead setting.

### Your answers

1. 
2. 
3. 
4. 
5. 
6. 
7. 
8. 
9. 
10. 
11. 

## References

- Hong, T., Pinson, P. and Fan, S. (2014) ‘Global Energy Forecasting Competition 2012’, *International Journal of Forecasting*, 30(2), pp. 357–363. https://doi.org/10.1016/j.ijforecast.2013.07.001
- Miller, C. et al. (2020) ‘The Building Data Genome Project 2, energy meter data from the ASHRAE Great Energy Predictor III competition’, *Scientific Data*, 7, 368. https://doi.org/10.1038/s41597-020-00712-x